# 01 — آماده‌سازی، پوشش و کیفیت داده مالی 

**هدف:** تبدیل ورودی مالی Freeze‌شده نسخه تاریخی به بازده‌های هفتگی سازگار با طرح ، بدون تغییر فایل‌های تاریخی.

این Notebook جمع‌آوری وب را تکرار نمی‌کند. ورودی‌های موجود را Hash، ممیزی و به خروجی تحلیل‌پذیر تبدیل می‌کند.

## قرارداد تحلیل

- بازه: `2026-02-28` تا `2026-07-22`
- هفته‌ها: `W01` تا `W21`؛ `W21` پنج‌روزه
- اصلی: `OIL_BRENT`, `GOLD_USD`, `VIX`, `IRR_USD`
- زمینه/حساسیت: `SP500`, `OIL_WTI`, `IRR_GOLD18`
- `TEDPIX`: فقط توصیفی
- متغیر هفتگی: مجموع بازده/تغییر لگاریتمی روزانه
- رویدادها: فقط `EV-016`, `EV-025`, `EV-031`

In [ ]:
from pathlib import Path

def find_project_root(start=None):
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "config" / "config.yaml").exists() and (candidate / "data" / "interim" / "financial" / "frozen_inputs").exists():
            return candidate
    raise FileNotFoundError("Project root with frozen financial inputs was not found.")

ROOT = find_project_root()
ROOT

## ۱. بارگذاری ماژول بازتولیدپذیر و ساخت همه خروجی‌ها

In [ ]:
import importlib.util
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Broken optional binary add-ons must not block the core pandas workflow.
for optional_module in ["pyarrow", "numexpr", "bottleneck"]:
    sys.modules.setdefault(optional_module, None)

import pandas as pd

MODULE_PATH = ROOT / "src" / "temporal_analysis" / "build_financial_outputs.py"
spec = importlib.util.spec_from_file_location("build_financial_outputs", MODULE_PATH)
financial = importlib.util.module_from_spec(spec)
spec.loader.exec_module(financial)

outputs = financial.run(ROOT)
{name: len(frame) for name, frame in outputs.items()}

## ۲. Inventory و Freeze

SHA-256 اجازه می‌دهد بعداً ثابت کنیم کدام نسخه فایل تاریخی مبنای خروجی  بوده است.

In [ ]:
outputs["financial_input_inventory_v1.csv"]

## ۳. تصمیم دارایی‌ها

نقش هر دارایی پیش از محاسبه همبستگی اجتماعی قفل می‌شود تا انتخاب شاخص براساس نتیجه انجام نشود.

In [ ]:
decisions = outputs["financial_asset_decisions_v1.csv"]
decisions[["asset_id", "analysis_tier", "analysis_use", "transform", "source", "instrument_type", "main_limitation"]]

## ۴. تبدیل روزانه به هفتگی

اگر `l_t` تغییر لگاریتمی روزانه باشد، تغییر لگاریتمی هفته مجموع `l_t`های مشاهده‌شده است. تغییر ساده از `exp(sum)-1` به دست می‌آید. روز فاقد معامله صفر یا Forward-fill نمی‌شود.

In [ ]:
weekly = outputs["financial_weekly_returns_v1.csv"]
weekly.head(12)

In [ ]:
weekly.groupby("asset_id", as_index=False).agg(
    n_weeks=("project_week", "nunique"),
    n_daily_returns=("n_return_obs", "sum"),
    minimum_weekly_change=("weekly_simple_change", "min"),
    maximum_weekly_change=("weekly_simple_change", "max"),
)

## ۵. پوشش

`return_coverage_ratio` تعداد بازده‌های مشاهده‌شده تقسیم بر روزهای باز مورد انتظار بازار است. آستانه ۸۰٪ یک قاعده Audit است، نه تضمین آماری.

In [ ]:
outputs["financial_coverage_summary_v1.csv"]

In [ ]:
coverage = outputs["financial_weekly_coverage_v1.csv"]
coverage.groupby(["asset_id", "coverage_status"]).size().rename("n_weeks").reset_index()

## ۶. پنجره توصیفی سه رویداد اصلی

تاریخ رویداد به نخستین بازده مشاهده‌شده در همان روز یا پس از آن نگاشت می‌شود. خروجی فقط هم‌زمانی را توصیف می‌کند و آزمون علّی نیست.

In [ ]:
outputs["financial_primary_event_windows_v1.csv"]

## ۷. کنترل‌های کیفیت

In [ ]:
checks = outputs["financial_quality_checks_v1.csv"]
checks

In [ ]:
assert checks["passed"].all(), "At least one financial quality check failed."
print(f"All {len(checks)} registered quality checks passed.")
print("Social-financial correlations remain pending until final weekly social outcomes exist.")